# Implementasi Bab 10 — Clustering

Segmentasi pelanggan sintetis dengan K-Means, Agglomerative Clustering, dan DBSCAN, disertai silhouette score dan pencarian k [ML-MODUL-01, hlm. 69–70].


In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
rng = np.random.default_rng(SEED)
X = np.vstack([
    rng.normal([5, 200_000], [1.5, 30_000], (100, 2)),
    rng.normal([15, 800_000], [2, 50_000], (100, 2)),
    rng.normal([2, 50_000], [0.8, 15_000], (100, 2)),
])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Bentuk data:", X_scaled.shape)


In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=SEED)
labels_kmeans = kmeans.fit_predict(X_scaled)
agglomerative = AgglomerativeClustering(n_clusters=3, linkage="ward")
labels_agglomerative = agglomerative.fit_predict(X_scaled)
dbscan = DBSCAN(eps=0.3, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_scaled)

kmeans_silhouette = silhouette_score(X_scaled, labels_kmeans)
agglomerative_silhouette = silhouette_score(X_scaled, labels_agglomerative)
dbscan_mask = labels_dbscan != -1
dbscan_cluster_count = len(set(labels_dbscan[dbscan_mask]))
dbscan_silhouette = (
    silhouette_score(X_scaled[dbscan_mask], labels_dbscan[dbscan_mask])
    if dbscan_cluster_count >= 2 else np.nan
)
cluster_summary = pd.DataFrame([
    ("K-Means", 3, kmeans_silhouette, 0),
    ("Hierarchical", 3, agglomerative_silhouette, 0),
    ("DBSCAN", dbscan_cluster_count, dbscan_silhouette, int((labels_dbscan == -1).sum())),
], columns=["model", "cluster_count", "silhouette", "noise_count"])
selected_business_model = cluster_summary.sort_values(
    "silhouette", ascending=False, na_position="last"
).iloc[0]["model"]
display(cluster_summary)
print("Rekomendasi numerik awal:", selected_business_model)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for axis, labels, title in zip(
    axes,
    [labels_kmeans, labels_agglomerative, labels_dbscan],
    ["K-Means", "Hierarchical", "DBSCAN"],
):
    axis.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, cmap="tab10", s=20)
    axis.set_title(title)
    axis.set_xlabel("Frekuensi (scaled)")
    axis.set_ylabel("Nilai transaksi (scaled)")
plt.tight_layout()
plt.show()


## Latihan 1: Elbow dan silhouette

Inertia selalu turun saat k bertambah; silhouette membantu menilai kepadatan dan pemisahan cluster [ML-MODUL-01, hlm. 70].


In [ ]:
k_values = range(2, 11)
inertias = []
silhouettes = []
for k in k_values:
    candidate = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    labels = candidate.fit_predict(X_scaled)
    inertias.append(candidate.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(k_values), inertias, marker="o")
axes[0].set(title="Elbow Method", xlabel="k", ylabel="Inertia")
axes[1].plot(list(k_values), silhouettes, marker="o")
axes[1].set(title="Silhouette", xlabel="k", ylabel="Score")
plt.show()
print("k silhouette terbaik:", list(k_values)[int(np.argmax(silhouettes))])


In [ ]:
linked = linkage(X_scaled, method="ward")
plt.figure(figsize=(12, 4))
dendrogram(linked, truncate_mode="lastp", p=20)
plt.title("Dendrogram Ringkas")
plt.xlabel("Cluster gabungan")
plt.ylabel("Jarak")
plt.show()


## Interpretasi bisnis

`selected_business_model` memilih silhouette tertinggi sebagai rekomendasi numerik awal. K-Means cocok ketika cluster relatif bulat dan jumlahnya diketahui; hierarchical membantu membaca struktur bertingkat; DBSCAN mampu menandai pola padat dan noise. Keputusan bisnis akhir tetap harus mempertimbangkan kemudahan interpretasi segmen, jumlah noise, dan tujuan kampanye—bukan silhouette saja.
